In [ ]:
import numpy as np
from qutip import *
from scipy.linalg import sqrtm, eigvalsh
from scipy.stats import linregress
from numba import njit, prange
import pickle
import os
import gc

In [ ]:
%matplotlib ipympl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from IPython.display import Image, display, Math

## Fidelity
Generic definition : 
$$ \mathcal{F}\left( \rho, \sigma \right) = \left( Tr \left[ \sqrt{ \sqrt{\rho} \sigma \sqrt{\rho} }\right] \right)^{2} $$ 
Definition for $ \rho $ Pure State and $ \sigma $ Mixed State : 
$$ \mathcal{F}\left( \rho, \sigma \right) = \langle \psi_{\rho} | \sigma | \psi_{\rho} \rangle $$
Definition for Pure State : 
$$ \mathcal{F}\left( \rho, \sigma \right) = |\langle \psi_{\rho} | \psi_{\sigma} \rangle|^{2} $$
Definition for Qubits : 
$$ \mathcal{F}\left( \rho, \sigma \right) = Tr\left( \rho \, \sigma \right) + 2 \sqrt{Det\left ( \rho \right) \, Det\left ( \sigma \right)} $$
## Trace Distance
Generic definition : 
$$ \mathcal{T}\left( \rho, \sigma \right) = \frac{1}{2} Tr \left[ \sqrt{\left( \rho - \sigma \right)^{\dagger} \left( \rho - \sigma  \right)} \right] $$
### Relationship : Fuchs-van de Graaf inequality
$$ 1 - \sqrt{\mathcal{F}\left( \rho, \sigma \right)} \leq \mathcal{T}\left( \rho, \sigma \right) \leq \sqrt{1 - \mathcal{F}\left( \rho, \sigma \right)} $$

In [ ]:
def fidelity_generic(rho, sigma):
    """
    Calculate the quantum fidelity between two generic density matrices.
    Formula: F(rho, sigma) = ( Tr[ sqrt( sqrt(rho) * sigma * sqrt(rho) ) ] )^2
    
    This version avoids scipy.linalg.sqrtm to prevent RuntimeWarnings, 
    using stable eigenvalue decomposition instead.
    
    Parameters:
        rho (numpy.ndarray): First density matrix (NxN).
        sigma (numpy.ndarray): Second density matrix (NxN).
        
    Returns:
        float: The fidelity between rho and sigma (real number between 0 and 1).
    """
    # 1. Square root of rho using eigenvalue decomposition
    evals_rho, evecs_rho = np.linalg.eigh(rho)
    # Truncate any negative noise to 0.0 before taking the square root
    evals_rho = np.maximum(evals_rho, 0.0) 
    sqrt_rho = evecs_rho @ np.diag(np.sqrt(evals_rho)) @ evecs_rho.conj().T
    
    # 2. Inner matrix: sqrt(rho) * sigma * sqrt(rho)
    inner_matrix = sqrt_rho @ sigma @ sqrt_rho
    
    # Force exact Hermiticity to remove any small imaginary noise
    inner_matrix = 0.5 * (inner_matrix + inner_matrix.conj().T)
    
    # 3. Trace of the square root is the sum of the square roots of the eigenvalues
    evals_inner = eigvalsh(inner_matrix)
    # Again, truncate negative noise to 0.0 before square root
    evals_inner = np.maximum(evals_inner, 0.0)
    
    fidelity = np.sum(np.sqrt(evals_inner))**2
    
    # Ensure numerical errors do not push fidelity slightly above 1.0
    return min(1.0, fidelity)
    

In [ ]:
@njit
def fidelity_qubit(rho, sigma):
    """
    Calculate the exact quantum fidelity between two single-qubit (2x2) density matrices.
    Formula: F(rho, sigma) = Tr(rho * sigma) + 2 * sqrt(Det(rho) * Det(sigma))
    """
    # Trace of the matrix product
    tr_term = np.real(np.trace(rho @ sigma))
    
    # Determinants of the two density matrices
    det_rho = np.real(np.linalg.det(rho))
    det_sigma = np.real(np.linalg.det(sigma))
    
    # FIX NUMERICO: Tronchiamo a 0 gli eventuali valori negativi infinitesimi
    det_rho = max(0.0, det_rho)
    det_sigma = max(0.0, det_sigma)
    
    # Calculate fidelity using the analytical formula for qubits
    fidelity = tr_term + 2.0 * np.sqrt(det_rho * det_sigma)
    
    return fidelity

In [ ]:
@njit
def fidelity_qubit_single_term(p00, p11, c10, c01, sigma):
    """
    Build the density matrix from its elements and then
    Calculate the exact quantum fidelity between two single-qubit (2x2) density matrices.
    Formula: F(rho, sigma) = Tr(rho * sigma) + 2 * sqrt(Det(rho) * Det(sigma))
    """
    # 1. Density Matrix build up 
    rho = np.zeros((2, 2), dtype=np.complex128)
    rho[0,0] = p00
    rho[0,1] = c01
    rho[1,0] = c10
    rho[1,1] = p11

    # 2. Trace of the matrix product (Calculated explicitly for 2x2 to avoid Numba issues)
    prod = rho @ sigma
    tr_term = prod[0,0].real + prod[1,1].real
    
    # 3. Determinants of the two 2x2 density matrices (ad - bc)
    det_rho = (rho[0,0] * rho[1,1] - rho[0,1] * rho[1,0]).real
    det_sigma = (sigma[0,0] * sigma[1,1] - sigma[0,1] * sigma[1,0]).real
    
    # 4. Numerical FIX: Truncate to 0 any infinitesimal negative values
    det_rho = max(0.0, det_rho)
    det_sigma = max(0.0, det_sigma)
    
    # 5. Calculate fidelity using the analytical formula for qubits
    fidelity = tr_term + 2.0 * np.sqrt(det_rho * det_sigma)
    
    return fidelity

In [ ]:

def trace_distance_generic(rho, sigma):
    """
    Calculate the Trace Distance between two generic density matrices.
    Formula: T(rho, sigma) = 1/2 * Tr[ sqrt( (rho - sigma)^dagger * (rho - sigma) ) ]
    
    Parameters:
        rho (numpy.ndarray): First density matrix (NxN).
        sigma (numpy.ndarray): Second density matrix (NxN).
        
    Returns:
        float: The trace distance between rho and sigma (real number between 0 and 1).
    """
    # Difference between the matrices
    diff = rho - sigma
    
    # Force exact Hermiticity to remove numerical noise
    diff = 0.5 * (diff + diff.conj().T)
    
    # Calculate the eigenvalues of the strictly Hermitian matrix 'diff'
    eigenvalues = eigvalsh(diff)
    
    # Trace distance is half the sum of the absolute eigenvalues
    t_dist = 0.5 * np.sum(np.abs(eigenvalues))
    
    # Ensure it stays within physical bounds
    return min(1.0, t_dist)
    

In [ ]:
@njit
def trace_distance_qubit(rho, sigma):
    """
    Calculate the exact Trace Distance between two single-qubit (2x2) density matrices.
    For a 2x2 traceless Hermitian matrix (rho - sigma), Det(diff) = -lambda^2 <= 0.
    Therefore, the Trace Distance simplifies to sqrt(-Det(rho - sigma)).
    
    Parameters:
        rho (numpy.ndarray): First density matrix (2x2).
        sigma (numpy.ndarray): Second density matrix (2x2).
        
    Returns:
        float: The trace distance between rho and sigma.
    """
    # Difference between the matrices
    diff = rho - sigma
    
    # Determinant of the difference
    det_diff = np.real(np.linalg.det(diff))
    
    # Since det_diff should be <= 0, -det_diff should be >= 0.
    # We use max(0.0, ...) to truncate any negative noise before applying sqrt.
    val_under_sqrt = max(0.0, -det_diff)
    
    t_dist = np.sqrt(val_under_sqrt)
    
    return min(1.0, t_dist)
    

In [ ]:
@njit
def trace_distance_qubit_single_term(p00, p11, c10, c01, sigma):
    """
    Build the density matrix from its elements and then
    Calculate the exact Trace Distance between two single-qubit (2x2) density matrices.
    For a 2x2 traceless Hermitian matrix (rho - sigma), Det(diff) = -lambda^2 <= 0.
    Therefore, the Trace Distance simplifies to sqrt(-Det(rho - sigma)).
    
    Parameters:
        rho (numpy.ndarray): First density matrix (2x2).
        sigma (numpy.ndarray): Second density matrix (2x2).
        
    Returns:
        float: The trace distance between rho and sigma.
    """
    # Density Matrix build up 
    rho = np.zeros((2, 2), dtype=np.complex128)
    rho[0,0] = p00
    rho[0,1] = c01
    rho[1,0] = c10
    rho[1,1] = p11
    
    # Difference between the matrices
    diff = rho - sigma
    
    # Determinant of the difference
    det_diff = (diff[0,0]*diff[1,1] - diff[0,1]*diff[1,0]).real
    
    # Since det_diff should be <= 0, -det_diff should be >= 0.
    # We use max(0.0, ...) to truncate any negative noise before applying sqrt.
    val_under_sqrt = max(0.0, -det_diff)
    
    t_dist = np.sqrt(val_under_sqrt)
    
    return min(1.0, t_dist)

In [ ]:
@njit
def compute_metrics_over_time(pop_10, pop_01, coh_1001, coh_0110, rho_lindblad_array):
    """
    Computes fidelity and trace distance over all time steps using Numba in C-speed.
    """
    time_steps = len(pop_10)
    
    fidelity_arr = np.zeros(time_steps)
    trace_dist_arr = np.zeros(time_steps)
    
    for t in range(time_steps):
        fidelity_arr[t] = fidelity_qubit_single_term(
            pop_10[t], pop_01[t], coh_1001[t], coh_0110[t], rho_lindblad_array[t]
        )
        trace_dist_arr[t] = trace_distance_qubit_single_term(
            pop_10[t], pop_01[t], coh_1001[t], coh_0110[t], rho_lindblad_array[t]
        )
        
    return fidelity_arr, trace_dist_arr

## Results Analysis

In [ ]:
# ====================================
# Physical & Simulation Parameters
# ====================================

# Site selector: 0 for |10>, 1 for |01>
site_index = 0

# Time parameters
dt = 0.01
tf = 100.0
time_steps = int(tf / dt)

# List of Number of trajectories to analyze
N_traj_list = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 20000]        

In [ ]:
# ============================================================
# GLOBAL CONFIGURATION :    
#   'normal'       → results_intermediate.pkl
#   'close_to_90'  → results_close_to_90_deg.pkl
# ============================================================

# ============================================================
# GLOBAL CONFIGURATION
# ============================================================

MODE = 'close_to_90'   # Switch this to 'normal' when you want to revert

# Configuration mapping based on MODE
_cfg = {
    'normal': {
        'Input_dir': "../Results/Data/Complete_rho/normal",
        'thetas_deg': [0, 30, 45, 60, 90],
        'Output_dir': "../Results/Plot/Fidelity/Avg/normal" 
    },
    'close_to_90': {
        'Input_dir': "../Results/Data/Complete_rho/close_90_deg",
        'thetas_deg': [0, 90, 89.9, 89.7, 89.5, 89, 88.5, 88, 87, 86],
        'Output_dir': "../Results/Plot/Fidelity/Avg/close_to_90"
    },
}

# Apply the selected configuration
cfg = _cfg[MODE]

# Set the global Output_dir dynamically based on the current mode
Output_dir = cfg['Output_dir']

# Create the output directory if it doesn't exist
os.makedirs(Output_dir, exist_ok=True)

print(f"--- Configuration Setup ---")
print(f"Current mode : {MODE}")
print(f"Angles list  : {cfg['thetas_deg']}")
print(f"Input Data   : {cfg['Input_dir']}")
print(f"Output Plots : {Output_dir}")

In [ ]:
# ===========================
# General Setup for Plotting
# ===========================

# Global Style Settings (Matplotlib rcParams)
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'figure.figsize': (10, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': ':',
    'figure.autolayout': True # plt.tight_layout()
})

# Automatic Figure Saving Function
def save_fig(fig, filename):
    """
    Saves the figure in both PNG or PDF
    """
    path_png = os.path.join(Output_dir, f"{filename}.png")
    # path_pdf = os.path.join(Output_dir, f"{filename}.pdf")  # save in pdf
    
    fig.savefig(path_png, dpi=300, bbox_inches='tight')
    # fig.savefig(path_pdf, bbox_inches='tight') # save in pdf
    print(f"Figure saved in: {Output_dir}/{filename}")

### Data Extraction

In [ ]:
# ====================
# Lindblad Extraction
# ====================

print("Starting Lindblad data extraction")

# Single extraction for Lindbad data, since they don't depend on theta !

# Format theta and dt for filename 
Lindblad_theta_rad = np.deg2rad(90)  # Always 90° for Lindblad

Lindblad_theta_str = f"{Lindblad_theta_rad:.6f}".replace(".", "p")
Lindblad_dt_str = f"{dt:.6f}".replace(".", "p")

 # File name
Lindblad_filename = f"result_theta{Lindblad_theta_str}_dt{Lindblad_dt_str}_Ntraj20000.npz"
Lindblad_filepath = os.path.join(cfg['Input_dir'], Lindblad_filename)

Lindblad_data = np.load(Lindblad_filepath)

rho_lindblad_complete = Lindblad_data['rho_list_lindblad']  # 4x4 dimension

rho_lindblad = np.zeros((time_steps, 2, 2), dtype=np.complex128)

for t in range(time_steps):
    
    # Populations : Index (2,2) -> |10><10|, Index (1,1)  -> |01><01| INVERTED respect to Trajectories, already inverted here
    rho_lindblad[t, 0, 0] = rho_lindblad_complete[t, 2, 2]  # |10><10|
    rho_lindblad[t, 1, 1] = rho_lindblad_complete[t, 1, 1]  # |01><01|
    
    # Coherences ATTENTION : inverted respect to Trajectories, already inverted here
    rho_lindblad[t, 0, 1] = rho_lindblad_complete[t, 2, 1]  # |10><01|
    rho_lindblad[t, 1, 0] = rho_lindblad_complete[t, 1, 2]  # |01><10|

print(" Lindblad data extraction completed")


In [ ]:
# ================================
# TRAJECTORY DATA EXTRACTION LOOP
# ================================

# =============================================
# Dictionaries to store only the averaged data 
# =============================================

# Structure: avg_fidelity_dict[theta][N_traj] = 1D_array_of_averaged_time_evolution
avg_fidelity_dict = {}

avg_trace_dist_dict = {} 

for theta in cfg['thetas_deg']:
    print(f"\nProcessing angle: {theta}°...")

    if theta not in avg_fidelity_dict:
        avg_fidelity_dict[theta] = {}
        avg_trace_dist_dict[theta] = {} 

    theta_rad = np.deg2rad(theta)
    theta_str = f"{theta_rad:.6f}".replace(".", "p")
    dt_str = f"{dt:.6f}".replace(".", "p")

    filename = f"result_theta{theta_str}_dt{dt_str}_Ntraj20000.npz"
    filepath = os.path.join(cfg['Input_dir'], filename)

    data = np.load(filepath)

    pop_traj_10 = data['pop_00']
    pop_traj_01 = data['pop_11']
    cohe_traj_10_01 = data['coh_10_01'] 
    cohe_traj_01_10 = data['coh_01_10']

    print(f"Data extraction for {theta}° completed.")
    print(f"Starting average, fidelity & trace distance calculation for {theta}°")

    for N in N_traj_list:
        pop_10_m   = np.mean(pop_traj_10[:, :N], axis=1)
        pop_01_m   = np.mean(pop_traj_01[:, :N], axis=1)
        coh_1001_m = np.mean(cohe_traj_10_01[:, :N], axis=1)
        coh_0110_m = np.mean(cohe_traj_01_10[:, :N], axis=1)

        fid_results, td_results = compute_metrics_over_time(
            pop_10_m, pop_01_m, coh_1001_m, coh_0110_m, rho_lindblad
        )
        
        avg_fidelity_dict[theta][N] = fid_results
        avg_trace_dist_dict[theta][N] = td_results

    data.close()
    del pop_traj_10, pop_traj_01, cohe_traj_10_01, cohe_traj_01_10, data
    gc.collect() 
    
    print(f"Angle {theta}° fully processed and memory cleared.")

## Fidelity calculation

In [ ]:
# ===================================================
# MAX & MEAN INFIDELITY & TRACE DISTANCE CALCULATION 
# ===================================================

print("Starting Infidelity metrics calculation...")

# Dictionaries to store the calculated metrics for plotting
max_infidelity_results = {}
mean_infidelity_results = {}
max_trace_distance_results = {}
mean_trace_distance_results = {}

for theta in cfg['thetas_deg']:
    max_vals_fidelity = []
    mean_vals_fidelity = []
    max_vals_trace_distance = []
    mean_vals_trace_distance = []
    
    for N in N_traj_list:
        # Extract the averaged time-evolution fidelity & trace distance arrays for this (theta, N)
        fid_time_array = avg_fidelity_dict[theta][N]
        td_time_array = avg_trace_dist_dict[theta][N]
        
        # Calculate infidelity: 1 - Fidelity
        infidelity_array = 1.0 - fid_time_array
        
        # Calculate the maximum and temporal mean of the infidelity
        max_inf = np.max(infidelity_array)
        mean_inf = np.mean(infidelity_array)
        max_td = np.max(td_time_array)
        mean_td = np.mean(td_time_array)
        
        max_vals_fidelity.append(max_inf)
        mean_vals_fidelity.append(mean_inf)
        max_vals_trace_distance.append(max_td)
        mean_vals_trace_distance.append(mean_td)

    # Convert lists to NumPy arrays for plotting
    max_infidelity_results[theta] = np.array(max_vals_fidelity)
    mean_infidelity_results[theta] = np.array(mean_vals_fidelity)
    max_trace_distance_results[theta] = np.array(max_vals_trace_distance)
    mean_trace_distance_results[theta] = np.array(mean_vals_trace_distance)

print("Infidelity max and mean values computed successfully.")

## Plot

In [ ]:
%matplotlib ipympl
from IPython.display import Image, display
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from ipywidgets import interact, IntSlider
import matplotlib.ticker as ticker

In [ ]:
plt.close('all')
plt.plot(avg_fidelity_dict[90][10000])
plt.show()

In [ ]:
plt.close('all')
plt.plot(avg_trace_dist_dict[90][10000])
plt.show()

### Fidelity average over time for different Number of Trajectories

In [ ]:
# ==========================
# MEAN INFIDELITY VS N_TRAJ
# ==========================

plt.close('all')

plt.figure()

# Cycle over all angles and plot the corresponding curves
for theta in cfg['thetas_deg']:
    
    # Retrieve the 1D array of mean infidelity values for this angle
    y_values_mean = mean_infidelity_results[theta]
    
    # Plot the curve for this specific angle
    plt.plot(N_traj_list, y_values_mean, marker='o', linestyle='-', 
             label=f'$\\theta$ = {theta}°')

# Formatting the plot
plt.xlabel(r"$\log(N_{\text{traj}})$")
plt.ylabel(r"$\log(1 - \langle \mathcal{F} \rangle_t)$")
plt.title(r"Convergence of $(1 - \langle \mathcal{F} \rangle_t)$ vs Trajectories")

# Logarithmic scale for both axes
plt.xscale('log')
plt.yscale('log')

# Grid and Legend
plt.grid(True, which="both", linestyle='--', alpha=0.5)
plt.legend()

# Save the figure using your global function
filename = f"Avg_Infidelity_in_time_{MODE}"
save_fig(plt.gcf(), filename)


plt.show()

### Minimal Fidelity for different Number of Trajectories

In [ ]:
# =========================
# MAX INFIDELITY VS N_TRAJ
# =========================

plt.figure()

# Cycle over all angles and plot the corresponding curves
for theta in cfg['thetas_deg']:
    
    # Retrieve the 1D array of max infidelity values for this angle
    y_values_max = max_infidelity_results[theta]
    
    # Plot the curve for this specific angle
    plt.plot(N_traj_list, y_values_max, marker='s', linestyle='-', 
             label=f'$\\theta$ = {theta}°')

# Formatting the plot
plt.xlabel(r"$\log(N_{\text{traj}})$")
plt.ylabel(r"$\log((1 - \mathcal{F})_{\max})$")
plt.title(r"Convergence of $(1 - \mathcal{F})_{\max}$ vs Trajectories")

# Logarithmic scale for both axes
plt.xscale('log')
plt.yscale('log')

# Grid and Legend
plt.grid(True, which="both", linestyle='--', alpha=0.5)
plt.legend()

# Save the figure using your global function
filename = f"Max_Infidelity_{MODE}"
save_fig(plt.gcf(), filename)

plt.show()

### Trace Distance average over time for different Number of Trajectories

In [ ]:
# ==============================
# MEAN TRACE DISTANCE VS N_TRAJ
# ==============================

plt.close('all')

plt.figure()

# Cycle over all angles and plot the corresponding curves
for theta in cfg['thetas_deg']:
    
    # Retrieve the 1D array of mean trace distance values for this angle
    y_values_mean = mean_trace_distance_results[theta]
    
    # Plot the curve for this specific angle
    plt.plot(N_traj_list, y_values_mean, marker='o', linestyle='-', 
             label=f'$\\theta$ = {theta}°')

# Formatting the plot
plt.xlabel(r"$\log(N_{\text{traj}})$")
plt.ylabel(r"$\log(\langle T \rangle_t)$")
plt.title(r"Convergence of $\langle T \rangle_t$ vs Trajectories")

# Logarithmic scale for both axes
plt.xscale('log')
plt.yscale('log')

# Grid and Legend
plt.grid(True, which="both", linestyle='--', alpha=0.5)
plt.legend()

# Save the figure using your global function
filename = f"Avg_Trace_Distance_in_time_{MODE}"
save_fig(plt.gcf(), filename)


plt.show()

### Maximal Trace Distance for different Number of Trajectories

In [ ]:
# =============================
# MAX TRACE DISTANCE VS N_TRAJ
# =============================

plt.figure()

# Cycle over all angles and plot the corresponding curves
for theta in cfg['thetas_deg']:
    
    # Retrieve the 1D array of max trace distance values for this angle
    y_values_max = max_trace_distance_results[theta]
    
    # Plot the curve for this specific angle
    plt.plot(N_traj_list, y_values_max, marker='s', linestyle='-', 
             label=f'$\\theta$ = {theta}°')

# Formatting the plot
plt.xlabel(r"$\log(N_{\text{traj}})$")
plt.ylabel(r"$\log(T_{\max})$")
plt.title(r"Convergence of $T_{\max}$ vs Trajectories")

# Logarithmic scale for both axes
plt.xscale('log')
plt.yscale('log')

# Grid and Legend
plt.grid(True, which="both", linestyle='--', alpha=0.5)
plt.legend()

# Save the figure using your global function
filename = f"Max_Trace_Distance_{MODE}"
save_fig(plt.gcf(), filename)

plt.show()

## Data Fitting

### Fidelity Avg in time

In [ ]:
# ===========================================
# GLOBAL FIT & PLOT: MEAN INFIDELITY in TIME
# ===========================================
from scipy.stats import linregress

# 1. Gather all valid data points (excluding exact zeros for log10)
all_N_inf = []
all_infidelity = []

for theta in cfg['thetas_deg']:
    mean_inf = mean_infidelity_results[theta]
    
    for N, inf in zip(N_traj_list, mean_inf):
        if inf > 0:  # Must be strictly positive for log10
            all_N_inf.append(N)
            all_infidelity.append(inf)

# 2. Convert to log10 scale
log_N_inf = np.log10(all_N_inf)
log_inf = np.log10(all_infidelity)

# 3. Perform Linear Regression
slope_inf, intercept_inf, r_value_inf, p_value_inf, std_err_inf = linregress(log_N_inf, log_inf)

print(f"--- Global Fit Results: Mean Infidelity ---")
print(f"Slope (m):     {slope_inf:.4f}  
print(f"Intercept (q): {intercept_inf:.4f}")
print(f"R-squared:     {r_value_inf**2:.4f}\n")

# 4. Plotting
plt.figure()

# Scatter plot of all raw data points from all angles
plt.scatter(log_N_inf, log_inf, color='blue', alpha=0.4, label='Data (all angles)')

# Theoretical fit line construction
x_fit_inf = np.linspace(min(log_N_inf), max(log_N_inf), 100)
y_fit_inf = slope_inf * x_fit_inf + intercept_inf

# Plot the fit line
plt.plot(x_fit_inf, y_fit_inf, color='red', linewidth=2, 
         label=f'Linear Fit: $y = {slope_inf:.2f}x {intercept_inf:+.2f}$')

# Formatting the plot
plt.xlabel(r"$\log_{10}(N_{traj})$")
plt.ylabel(r"$\log_{10}(\langle 1 - \mathcal{F} \rangle_t)$")
plt.title("Global Scaling Fit: Mean Infidelity vs Trajectories")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

# Save the figure dynamically using the global function
#save_fig(plt.gcf(), f"Global_Fit_Mean_Infidelity_{MODE}")

plt.show()

### Trace Distance FittinAvg in time

In [ ]:
# ===============================================
# GLOBAL FIT & PLOT: MEAN TRACE DISTANCE in TIME
# ===============================================

# 1. Gather all valid data points (excluding exact zeros for log10)
all_N_td = []
all_td = []

for theta in cfg['thetas_deg']:
    mean_td = mean_trace_distance_results[theta] # Make sure this matches your variable name!
    
    for N, td in zip(N_traj_list, mean_td):
        if td > 0:  # Must be strictly positive for log10
            all_N_td.append(N)
            all_td.append(td)

# 2. Convert to log10 scale
log_N_td = np.log10(all_N_td)
log_td = np.log10(all_td)

# 3. Perform Linear Regression
slope_td, intercept_td, r_value_td, p_value_td, std_err_td = linregress(log_N_td, log_td)

print(f"--- Global Fit Results: Mean Trace Distance ---")
print(f"Slope (m):     {slope_td:.4f} 
print(f"Intercept (q): {intercept_td:.4f}")
print(f"R-squared:     {r_value_td**2:.4f}\n")

# 4. Plotting
plt.figure()

# Scatter plot of all raw data points from all angles
plt.scatter(log_N_td, log_td, color='green', alpha=0.4, label='Data (all angles)')

# Theoretical fit line construction
x_fit_td = np.linspace(min(log_N_td), max(log_N_td), 100)
y_fit_td = slope_td * x_fit_td + intercept_td

# Plot the fit line
plt.plot(x_fit_td, y_fit_td, color='red', linewidth=2, 
         label=f'Linear Fit: $y = {slope_td:.2f}x {intercept_td:+.2f}$')

# Formatting the plot
plt.xlabel(r"$\log_{10}(N_{traj})$")
plt.ylabel(r"$\log_{10}(\langle \mathcal{T} \rangle_t)$")
plt.title("Global Scaling Fit: Mean Trace Distance vs Trajectories")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

# Save the figure dynamically using the global function
#save_fig(plt.gcf(), f"Global_Fit_Mean_Trace_Distance_{MODE}")

plt.show()

## Trace Distance Fit for different theta

In [ ]:
# ========================================================
# ANGLE-BY-ANGLE FIT & PLOT: MEAN TRACE DISTANCE in TIME
# ========================================================
from scipy.stats import linregress
import numpy as np
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

print("--- Angle-by-Angle Fit Results: Mean Trace Distance ---")

# Create a color map to easily distinguish different angles in the plot
colors = plt.cm.viridis(np.linspace(0, 0.9, len(cfg['thetas_deg'])))

# Loop over all angles to perform individual fits
for idx, theta in enumerate(cfg['thetas_deg']):
    mean_td = mean_trace_distance_results[theta] 
    
    valid_N_td = []
    valid_td = []
    
    # 1. Gather valid data points for THIS specific angle
    for N, td in zip(N_traj_list, mean_td):
        if td > 0:  # Must be strictly positive for log10
            valid_N_td.append(N)
            valid_td.append(td)

    # 2. Convert to log10 scale
    log_N_td = np.log10(valid_N_td)
    log_td = np.log10(valid_td)

    # 3. Perform Linear Regression for THIS angle
    slope_td, intercept_td, r_value_td, p_value_td, std_err_td = linregress(log_N_td, log_td)

    # Print the exact equation and R-squared for the current angle
    eq_string = f"y = {slope_td:.4f}x {intercept_td:+.4f}"
    print(f"Theta: {theta:5.1f}° | Equation: {eq_string} | R-squared: {r_value_td**2:.4f}")

    # 4. Plotting
    color = colors[idx]
    
    # Scatter plot of raw data points for this angle
    plt.scatter(log_N_td, log_td, color=color, alpha=0.4, marker='o')

    # Theoretical fit line construction
    x_fit_td = np.linspace(min(log_N_td), max(log_N_td), 100)
    y_fit_td = slope_td * x_fit_td + intercept_td

    # Plot the fit line and include the equation in the legend
    plt.plot(x_fit_td, y_fit_td, color=color, linewidth=2, 
             label=f'$\\theta$ = {theta}°: $y = {slope_td:.2f}x {intercept_td:+.2f}$')

# Formatting the plot
plt.xlabel(r"$\log_{10}(N_{traj})$")
plt.ylabel(r"$\log_{10}(\langle \mathcal{T} \rangle_t)$")
plt.title("Scaling Fit per Angle: Mean Trace Distance vs Trajectories")

# Place the legend outside the plot area so it doesn't overlap the lines
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

# Save the figure dynamically using the global function (uncomment when needed)
# save_fig(plt.gcf(), f"Angle_Fits_Mean_Trace_Distance_{MODE}")

plt.show()